# 06 — v2.5 · The network between the 23 complexes (run spec v3 §1a revised 2026-09-29; §3 stages 3–4, §4)

`cc.start` on the headline variant + the contraction overrides, `run_id` from config (`v25_run001`; a NEW id after a second-pass
merge). The nodes are the complexes; inter-complex links are routed between the complex UNIONS (a path may leave any patch of
the source and enter any patch of the target — the unit field is the pointwise minimum over the member patches' cached fields,
so the CWD stage is a cache HIT). Order: G0 (n complexes) + GW5 → resistance → CWD → cutoff (inherited, D31) → network (G3) →
GW4 → G15 → near-optimality (G10/G22) → the counterfactual (D17, G13; cache HIT on the patch fields) → the **D25 second pass**
(merge + re-run if any inter-complex link is near-contiguous) → branches (D26; G9/G21) → `classify_links` (D23–D25; G18/G19) →
protection status (W11) → priority → `finish` (Layer A written by write_run). ~1 h; live internet not needed.

In [1]:
# ---- Setup: find the project root, import the shared engines ----------------
# This notebook lives in analyses/wolverine_refugia_connectivity/, below the repo root where
# config.py and the corridor engine modules sit. Same bootstrap as analyses/northern_connectivity/.
import sys, pathlib, json
_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
          if (p / "config.py").exists()]
assert _cands, f"config.py not found above {pathlib.Path.cwd()} -- run this notebook from inside the repo"
ROOT = _cands[0]
sys.path.insert(0, str(ROOT))

import importlib
import numpy as np
import pandas as pd
import config
import corridors_prep as cp
import corridor_graph as cg
import corridors_core as cc
for _m in (config, cp, cg, cc):
    importlib.reload(_m)

KEY = "wolverine"
CFG = config.CORRIDORS[KEY]
HERE = ROOT / "analyses" / "wolverine_refugia_connectivity"

In [2]:
V25 = CFG["v25"]
OV = cc._merge(CFG["variants"][CFG["headline_variant"]], V25["overrides"])
RUN_ID = V25["run_id"]                                      # <- bump after a second-pass merge (e.g. "v25_run002")
_ = cc.resolve(KEY, overrides=OV)                           # G8
A = cc.start(KEY, overrides=OV, label="v2.5: the network between the refugia complexes, withheld-terrain surface", run_id=RUN_ID, require_review=False)
summ = json.loads((HERE / "audit" / "audit_objects" / "complexes_summary.json").read_text())
cc.gate_g0(A, expect_names=summ["n_complexes"], expect_merges=0)
cc.gate_gw5(A)
ALLOW_DIRTY = False                                          # run spec v3 §9: the run record on a clean tree (True only for a smoke)
print("git:", A.rec["git"])
assert ALLOW_DIRTY or not A.rec["git"].get("dirty", True), "working tree dirty at run creation -- delete this run dir, commit, re-run (run spec v3 §9)"

run v25_run001 (v2.5: the network between the refugia complexes, withheld-terrain surface) -> output_data/corridors_wolverine/v25_run001
  NOTE working tree is dirty; the recorded git SHA does not fully describe this run
Y2Y — wolverine climate refugia: routing grid 4285x11038 @ 300 m = 17,240,740 routable cells (1,551,667 km²) -- whole window
  core refugia 318,822 km² in 1,845 8-connected patches; marginal 169,988 km²
    >=   100 km²:  239 patches holding  94.2% of core area
    >=   250 km²:  130 patches holding  88.8% of core area   <- node floor
    >=   500 km²:   67 patches holding  81.7% of core area
    >=  1000 km²:   36 patches holding  75.0% of core area
  node_names.csv applied: 130 display names overridden
  protected land (status layer, W11): existing PAs 201,672 km², proposed 179,528 km² | 43.5% of node land is protected
nodes: 130 core refugia patches >= 250 km² (282,996 km² of node land, numbered north -> south) | largest 78,454 km², smallest 254 km²
GW5 OK: 130 patc

AssertionError: working tree dirty at run creation -- delete this run dir, commit, re-run (run spec v3 §9)

## Resistance → CWD (cache HIT on the 130 patch fields; the complexes' fields derived) → the inherited cutoff

In [ ]:
cc.resistance(A)
cc.cost_distances(A)
cc.set_cutoff(A, CFG["cwd_cutoff_abs"])

## Network (G3) → GW4 → G15

In [ ]:
cc.corridor_network(A)
cc.gate_gw4(A)
cc.gate_g15(A)

## Near-optimality (G10, G22) → the counterfactual width (D17, G13)

In [ ]:
cc.near_optimality(A)
cc.counterfactual_squeeze(A)

## D25 second pass

If any inter-complex link is still near-contiguous, the pair is merged (`cc.merge_complexes`) and this notebook is re-run from the
top with a NEW `RUN_ID`. Expected zero or few.

In [ ]:
m2 = cc.second_pass_merges(A)
if len(m2):
    print(m2.to_string(index=False))
    print("-> cc.merge_complexes(KEY, [(i, j), ...]) with the pairs above; commit the audit objects; set RUN_ID to a new id; re-run this notebook")
assert not len(m2), "second-pass merges pending -- see above"

## Route branches (D26; G9/G21) → link classes (D23–D25; G18/G19) → protection status (W11) → priority → finish

In [ ]:
cc.route_branches(A)
tt = pd.read_csv(A.run_dir / "class_truth_table.csv"); print(tt.to_string(index=False))
cc.secured_status(A)
cc.priority_surface(A)
cc.finish(A)
cc.runs(KEY)

## Done — next `07_v25_product.ipynb` (coverage, accounting, tables, the headline) then `08_director_outputs.ipynb`.